In [26]:
import pandas as pd
import min_features, daily_return
import importlib
importlib.reload(min_features)
importlib.reload(daily_return)

perf_df = pd.read_csv("daily_w_prob.csv")
perf_df['Date'] = perf_df['test_start']
returns = [1, 2, 3, 5, 10]
df_daily = daily_return.pull_daily('QQQ', returns) 
return_cols = df_daily.columns[df_daily.columns.str.contains("Return_")].to_list()
df_returns = df_daily[['Date'] + return_cols][(df_daily['Date'] < '2025-12-20') & (df_daily['Date'] > '2025-01-01')].copy()

In [32]:
import numpy as np
import pandas as pd

def flip_bucket_tables_multi(
    df_daily,
    perf_df,
    returns,
    *,
    streak_col="streak_lag1",
    K=3,
    date_col="Date",
    close_col="Close",
    w=None,
):
    """
    For each horizon r:
      - builds streak_lag1 from Return_r (via day-to-day streak logic)
      - merges into perf_df rows for (horizon=r, test_days)
      - buckets streak_lag1 into [-K..K] plus tails as +/- (K+1) labeled "3+"
      - computes acc/n wide table and weighted balanced-accuracy score (wba)

    Returns:
      flip_by_r: dict[r] -> flip_wide (MultiIndex columns)
      rf_sorted_by_r: dict[r] -> sorted flip_wide by wba
      all_flip: concat of flip_wide with horizon as index level
    """
    if w is None:
        # weights: ±1 -> 2.0, ±2 -> 1.5, ±3 -> 1.25, ±3+ -> 1.0
        w = {1: 2.0, 2: 1.5, 3: 1.25, "3+": 1.0}

    max_score = sum(w.values())
    gcols = ["model", "train_years", "feature_set"]

    # --- helper: compute streak + streak_lag1 for a given Return_r ---
    def _add_streak(df_base, ret_col):
        d = df_base[[date_col, close_col, ret_col]].sort_values(date_col).copy()
        s = d[ret_col].astype("int8")
        grp = s.ne(s.shift()).cumsum()
        streak_len = s.groupby(grp).cumcount() + 1
        d["streak"] = streak_len.where(s.eq(1), -streak_len).astype("int32")
        d["streak_lag1"] = d["streak"].shift(1).fillna(0).astype("Int64")
        return d

    base = df_daily[[date_col, close_col] + [f"Return_{r}" for r in returns]].copy()

    flip_by_r = {}
    rf_sorted_by_r = {}

    for r in returns:
        ret_col = f"Return_{r}"

        # build streak_lag1
        df_r = _add_streak(base, ret_col)

        # merge with perf (this gives you acc/model/train_years/feature_set/etc)
        perf_r = perf_df[(perf_df["horizon"] == r)]
        d = df_r.merge(perf_r, on=date_col, how="inner")

        # --- bucket streak_lag1 into [-K..K] plus tails as +/- (K+1) ---
        d["streak_bucket"] = d[streak_col].clip(lower=-K, upper=K)
        d.loc[d[streak_col] < -K, "streak_bucket"] = -(K + 1)
        d.loc[d[streak_col] >  K, "streak_bucket"] =  (K + 1)

        flip_perf = (
            d.groupby(gcols + ["streak_bucket"], sort=False)
             .agg(n=("acc", "size"), acc=("acc", "mean"))
        )

        flip_wide = pd.concat(
            {"acc": flip_perf["acc"].unstack("streak_bucket"),
             "n":   flip_perf["n"].unstack("streak_bucket")},
            axis=1
        )

        # enforce column order: +1/-1, +2/-2, +3/-3, 3+/-3+
        ordered_cols = []
        for k in [1, 2, 3, "3+"]:
            pb = (K + 1) if k == "3+" else k
            nb = -(K + 1) if k == "3+" else -k
            ordered_cols += [("acc", pb), ("acc", nb), ("n", pb), ("n", nb)]
        flip_wide = flip_wide.reindex(columns=pd.MultiIndex.from_tuples(ordered_cols))

        # relabel buckets to strings ("3+", "-3+", etc.)
        rename_cols = []
        for metric, b in flip_wide.columns:
            if b == (K + 1): lab = "3+"
            elif b == -(K + 1): lab = "-3+"
            else: lab = str(b)
            rename_cols.append((metric, lab))
        flip_wide.columns = pd.MultiIndex.from_tuples(rename_cols)

        # --- per-pair bal_acc and weighted score ---
        def _pair_bal(pos_lab, neg_lab):
            return (flip_wide[("acc", pos_lab)] + flip_wide[("acc", neg_lab)]) / 2

        flip_wide[("bal_acc_pair", "1")]  = _pair_bal("1",  "-1")
        flip_wide[("bal_acc_pair", "2")]  = _pair_bal("2",  "-2")
        flip_wide[("bal_acc_pair", "3")]  = _pair_bal("3",  "-3")
        flip_wide[("bal_acc_pair", "3+")] = _pair_bal("3+", "-3+")

        # weighted SUM, normalized by max_score (your current behavior)
        flip_wide[("wba", "")] = (
            w[1]    * flip_wide[("bal_acc_pair", "1")] +
            w[2]    * flip_wide[("bal_acc_pair", "2")] +
            w[3]    * flip_wide[("bal_acc_pair", "3")] +
            w["3+"] * flip_wide[("bal_acc_pair", "3+")]
        ) / max_score

        flip_wide[("wba", "")] = flip_wide[("wba", "")].round(2)

        flip_by_r[r] = flip_wide
        rf_sorted_by_r[r] = flip_wide.sort_values(by=("wba", ""), ascending=False).round(2)

    all_flip = pd.concat(flip_by_r, names=["horizon"])
    return flip_by_r, rf_sorted_by_r, all_flip

returns = [2, 5, 10]
streak_col = "streak"
flip_by_r, rf_sorted_by_r, all_flip = flip_bucket_tables_multi(
    df_daily=df_daily,
    perf_df=perf_df,
    returns=returns,
    streak_col=streak_col,
    K=3,
)

rf_sorted_by_r[10]     # horizon 10 sorted table
flip_by_r[5]           # horizon 5 unsorted table
all_flip.round(2)               # all horizons stacked with "horizon" index level


acc         n       acc        \
                                                  1    -1   1  -1     2    -2   
horizon model         train_years feature_set                                   
2       xgboost       4           daily        0.42  0.15  36  34  0.74  0.62   
                                  d-no-skew    0.53  0.21  36  34  0.61  0.58   
                                  d-no-vix     0.39  0.24  36  34  0.71  0.58   
                      6           daily        0.53  0.18  36  34  0.74  0.67   
                                  d-no-skew    0.56  0.24  36  34  0.68  0.71   
                                  d-no-vix     0.50  0.21  36  34  0.81  0.67   
        random_forest 4           daily        0.31  0.12  36  34  0.77  0.71   
                                  d-no-skew    0.39  0.12  36  34  0.81  0.71   
                                  d-no-vix     0.42  0.09  36  34  0.84  0.71   
                      6           daily        0.50  0.06  36  34  0.84  0.54   
                                  d-no-skew    0.61  0.06  36  34  0.77  0.62   
                                  d-no-vix     0.50  0.12  36  34  0.87  0.58   
5       xgboost       4           daily        0.44  0.06  18  16  0.87  0.46   
                                  d-no-skew    0.50  0.06  18  16  0.93  0.54   
                                  d-no-vix     0.33  0.00  18  16  0.87  0.46   
                      6           daily        0.56  0.00  18  16  0.87  0.38   
                                  d-no-skew    0.50  0.00  18  16  0.93  0.31   
                                  d-no-vix     0.61  0.00  18  16  0.80  0.31   
        random_forest 4           daily        0.28  0.06  18  16  0.87  0.62   
                                  d-no-skew    0.28  0.06  18  16  0.87  0.69   
                                  d-no-vix     0.33  0.12  18  16  0.87  0.69   
                      6           daily        0.50  0.00  18  16  0.93  0.31   
                                  d-no-skew    0.50  0.00  18  16  0.93  0.38   
                                  d-no-vix     0.50  0.00  18  16  0.87  0.23   
10      xgboost       4           daily        0.36  0.00  11   9  0.70  0.38   
                                  d-no-skew    0.45  0.00  11   9  0.90  0.38   
                                  d-no-vix     0.55  0.11  11   9  0.50  0.38   
                      6           daily        0.55  0.00  11   9  0.60  0.50   
                                  d-no-skew    0.45  0.00  11   9  0.80  0.38   
                                  d-no-vix     0.36  0.00  11   9  0.80  0.38   
        random_forest 4           daily        0.36  0.00  11   9  0.80  0.50   
                                  d-no-skew    0.45  0.00  11   9  0.80  0.38   
                                  d-no-vix     0.27  0.00  11   9  0.60  0.38   
                      6           daily        0.45  0.00  11   9  0.70  0.25   
                                  d-no-skew    0.45  0.00  11   9  0.90  0.25   
                                  d-no-vix     0.55  0.00  11   9  0.80  0.25   

                                                n       acc        ...   n  \
                                                2  -2     3    -3  ...  -3   
horizon model         train_years feature_set                      ...       
2       xgboost       4           daily        31  24  0.80  0.64  ...  14   
                                  d-no-skew    31  24  0.90  0.64  ...  14   
                                  d-no-vix     31  24  0.90  0.79  ...  14   
                      6           daily        31  24  0.85  0.71  ...  14   
                                  d-no-skew    31  24  0.85  0.50  ...  14   
                                  d-no-vix     31  24  0.85  0.64  ...  14   
        random_forest 4           daily        31  24  0.85  0.71  ...  14   
                                  d-no-skew    31  24  0.80  0.50  ...  14   
                              

In [46]:
import numpy as np
import pandas as pd

def flip_bucket_tables_multi_dual(
    df_daily,
    perf_df,
    returns,
    *,
    K=3,
    date_col="Date",
    close_col="Close",
    w=None,
    perf_filter=None,  # optional callable to filter perf_df per horizon
):
    """
    For each horizon r:
      - builds streak + streak_lag1 from Return_r
      - merges into perf_df rows for (horizon=r) (and any extra filters you provide)
      - buckets BOTH streak and streak_lag1 into [-K..K] plus tails as +/- (K+1) labeled "3+"
      - computes:
          - wba_close (from streak) and wba_open (from streak_lag1)
          - bal_acc pair scores for +/-1, +/-2, +/-3 for both contexts
          - (optional) keeps acc/n wide columns for each context with suffixes _c and _o

    Returns:
      by_r: dict[r] -> wide table (flat columns)
      all_out: concat of all horizons with horizon as index level (flat columns)
    """
    if w is None:
        # weights: ±1 -> 2.0, ±2 -> 1.5, ±3 -> 1.25, ±3+ -> 1.0
        w = {1: 2.0, 2: 1.5, 3: 1.25, "3+": 1.0}

    max_score = float(sum(w.values()))
    gcols = ["model", "train_years", "feature_set"]

    def _add_streak(df_base, ret_col):
        d = df_base[[date_col, close_col, ret_col]].sort_values(date_col).copy()
        s = d[ret_col].astype("int8")
        grp = s.ne(s.shift()).cumsum()
        streak_len = s.groupby(grp).cumcount() + 1
        d["streak"] = streak_len.where(s.eq(1), -streak_len).astype("int32")
        d["streak_lag1"] = d["streak"].shift(1).fillna(0).astype("Int64")
        return d

    def _bucketize(series: pd.Series) -> pd.Series:
        b = series.clip(lower=-K, upper=K).astype("int32")
        b = b.copy()
        b.loc[series < -K] = -(K + 1)
        b.loc[series >  K] =  (K + 1)
        return b

    def _make_context_table(d: pd.DataFrame, bucket_col: str, suffix: str) -> pd.DataFrame:
        """
        suffix: "_c" for streak (close), "_o" for streak_lag1 (open)
        returns a flat-column wide df indexed by gcols, containing:
          - acc_{bucket}{suffix}, n_{bucket}{suffix} for buckets {1,-1,2,-2,3,-3,3+,-3+}
          - bal_acc_{1/2/3}{suffix}
          - wba_{close/open} (handled outside)
        """
        flip_perf = (
            d.groupby(gcols + [bucket_col], sort=False)
             .agg(n=("acc", "size"), acc=("acc", "mean"))
        )

        wide = pd.concat(
            {"acc": flip_perf["acc"].unstack(bucket_col),
             "n":   flip_perf["n"].unstack(bucket_col)},
            axis=1
        )

        # enforce order: +1/-1, +2/-2, +3/-3, 3+/-3+
        ordered_cols = []
        for k in [1, 2, 3, "3+"]:
            pb = (K + 1) if k == "3+" else k
            nb = -(K + 1) if k == "3+" else -k
            ordered_cols += [("acc", pb), ("acc", nb), ("n", pb), ("n", nb)]
        wide = wide.reindex(columns=pd.MultiIndex.from_tuples(ordered_cols))

        # relabel bucket keys to strings ("3+", "-3+", etc.)
        rename_cols = []
        for metric, b in wide.columns:
            if b == (K + 1): lab = "3+"
            elif b == -(K + 1): lab = "-3+"
            else: lab = str(b)
            rename_cols.append((metric, lab))
        wide.columns = pd.MultiIndex.from_tuples(rename_cols)

        # flatten to single-level names: acc_1_c, n_-2_o, etc.
        flat = wide.copy()
        flat.columns = [f"{m}_{b}{suffix}" for (m, b) in flat.columns]

        # pair bal_acc for +/-1,2,3 (no 3+ requested here)
        def _pair(acc_pos, acc_neg):
            return (acc_pos + acc_neg) / 2

        flat[f"bal_acc_1{suffix}"] = _pair(flat.get(f"acc_1{suffix}"),  flat.get(f"acc_-1{suffix}"))
        flat[f"bal_acc_2{suffix}"] = _pair(flat.get(f"acc_2{suffix}"),  flat.get(f"acc_-2{suffix}"))
        flat[f"bal_acc_3{suffix}"] = _pair(flat.get(f"acc_3{suffix}"),  flat.get(f"acc_-3{suffix}"))
        flat[f"bal_acc_3p{suffix}"] = _pair(flat.get(f"acc_3+{suffix}"), flat.get(f"acc_-3+{suffix}"))

        return flat

    base = df_daily[[date_col, close_col] + [f"Return_{r}" for r in returns]].copy()

    by_r = {}

    for r in returns:
        ret_col = f"Return_{r}"
        df_r = _add_streak(base, ret_col)

        # merge with perf
        perf_r = perf_df[perf_df["horizon"] == r]
        if perf_filter is not None:
            perf_r = perf_filter(perf_r)

        d = df_r.merge(perf_r, on=date_col, how="inner")

        # bucket both contexts
        d["bucket_c"] = _bucketize(d["streak"])       # close-context
        d["bucket_o"] = _bucketize(d["streak_lag1"])  # open-context

        tab_c = _make_context_table(d, "bucket_c", "_c")
        tab_o = _make_context_table(d, "bucket_o", "_o")

        # combine side-by-side
        out = tab_c.join(tab_o, how="outer")

        # compute wba_close / wba_open from each context’s pair balances
        out["wba_close"] = (
            w[1]   * out["bal_acc_1_c"] +
            w[2]   * out["bal_acc_2_c"] +
            w[3]   * out["bal_acc_3_c"] +
            w["3+"] * out["bal_acc_3p_c"]
        ) / max_score

        out["wba_open"] = (
            w[1]   * out["bal_acc_1_o"] +
            w[2]   * out["bal_acc_2_o"] +
            w[3]   * out["bal_acc_3_o"] +
            w["3+"] * out["bal_acc_3p_o"]
        ) / max_score

        out["wba_close"] = out["wba_close"].round(2)
        out["wba_open"]  = out["wba_open"].round(2)

        # optional: keep horizon as a column too (handy for later concat)
        out = out.reset_index()
        out.insert(0, "horizon", r)

        by_r[r] = out

    all_out = pd.concat(by_r.values(), ignore_index=True)
    return by_r, all_out


# ---- usage ----
returns = [2, 5, 10]
by_r, all_out = flip_bucket_tables_multi_dual(
    df_daily=df_daily,
    perf_df=perf_df,
    returns=returns,
    K=3,
)

# horizon 10 table
by_r[10].sort_values(["wba_close", "wba_open"], ascending=False).head(25)

# all horizons combined
perf_columns = ['horizon', 'model', 'train_years', 'feature_set',# 'acc_1_c', 'acc_-1_c', 'n_1_c', 'n_-1_c',
                 'bal_acc_1_c', 'bal_acc_2_c', 'bal_acc_3_c', 'bal_acc_3p_c',
                 'bal_acc_1_o', 'bal_acc_2_o', 'bal_acc_3_o', 'bal_acc_3p_o', 
                 'wba_close', 'wba_open']

# top per horizon (ranked by MCC desc, then Brier asc)
top_by_horizon = (
    all_out
    .sort_values(["horizon", 'wba_close', 'wba_open'], ascending=[True, False, False])
    .groupby("horizon", as_index=False, sort=False)
    .head(2)
)

top_by_horizon[perf_columns].round(2)

,horizon,model,train_years,feature_set,bal_acc_1_c,bal_acc_2_c,bal_acc_3_c,bal_acc_3p_c,bal_acc_1_o,bal_acc_2_o,bal_acc_3_o,bal_acc_3p_o,wba_close,wba_open
7,2,xgboost,4,d-no-vix,0.31,0.65,0.84,0.84,0.60,0.67,0.65,0.64,0.61,0.64
10,2,xgboost,6,d-no-vix,0.35,0.74,0.75,0.76,0.72,0.63,0.56,0.60,0.61,0.64
13,5,random_forest,4,d-no-vix,0.23,0.78,1.00,0.95,0.71,1.00,0.90,0.79,0.67,0.84
12,5,random_forest,4,d-no-skew,0.17,0.78,1.00,0.92,0.71,1.00,0.79,0.78,0.64,0.82
30,10,xgboost,4,d-no-skew,0.23,0.64,0.93,0.94,0.61,0.94,0.78,0.89,0.61,0.78
24,10,random_forest,4,d-no-skew,0.23,0.59,0.86,0.97,0.56,0.88,0.78,0.92,0.59,0.75


In [24]:
import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss, log_loss, matthews_corrcoef, balanced_accuracy_score

gcols = ["horizon", "model", "train_years", "feature_set"]

y_col = "test_pos_n"   # 0/1 actual
p_col = "pred"         # P(y=1)

def _clip01(p, eps=1e-15):
    p = np.asarray(p, dtype=float)
    return np.clip(p, eps, 1 - eps)

def _metrics(g: pd.DataFrame) -> pd.Series:
    y = g[y_col].astype(int).to_numpy()
    p = _clip01(g[p_col].to_numpy())

    # hard preds at 0.5
    yhat = (p >= 0.5).astype(int)

    # confident subset mask
    sel = (p >= 0.6) | (p <= 0.4)

    # probability metrics
    brier = brier_score_loss(y, p)
    ll = log_loss(y, p)

    # MCC (needs both classes in y and yhat)
    mcc = np.nan
    if (np.unique(y).size > 1) and (np.unique(yhat).size > 1):
        mcc = matthews_corrcoef(y, yhat)

    # Balanced accuracy (needs both classes in y)
    bal_acc = np.nan
    if np.unique(y).size > 1:
        bal_acc = balanced_accuracy_score(y, yhat)

    # confident accuracy + coverage
    cov = float(sel.mean())
    acc_conf = float((yhat[sel] == y[sel]).mean()) if sel.any() else np.nan

    return pd.Series({
        "pos_rate": float(y.mean()),
        "bal_acc": float(bal_acc) if not np.isnan(bal_acc) else np.nan,
        "brier": float(brier),
        "log_loss": float(ll),
        "mcc": float(mcc) if not np.isnan(mcc) else np.nan,
        "cov_|0.6|": cov,
        "acc_|0.6|": acc_conf,
    })

metrics_df = (
    perf_df
    .dropna(subset=[y_col, p_col])
    .groupby(gcols, sort=False)
    .apply(_metrics, include_groups=False)
    .reset_index()
    .sort_values(["horizon", "mcc", "brier"], ascending=[True, False, True])
)

# top per horizon (ranked by MCC desc, then Brier asc)
top_by_horizon = (
    metrics_df
    .sort_values(["horizon", "bal_acc", "mcc", "brier"], ascending=[True, False, False, True])
    .groupby("horizon", as_index=False, sort=False)
    .head(2)
)

top_by_horizon.round(2)

,horizon,model,train_years,feature_set,pos_rate,bal_acc,brier,log_loss,mcc,cov_|0.6|,acc_|0.6|
26,2,xgboost,6,d-no-vix,0.58,0.63,0.28,3.20,0.27,0.92,0.67
24,2,xgboost,4,d-no-vix,0.58,0.63,0.27,3.44,0.26,0.92,0.65
29,5,random_forest,4,d-no-vix,0.64,0.77,0.16,0.50,0.57,0.82,0.87
5,5,random_forest,4,daily,0.64,0.77,0.16,0.51,0.56,0.82,0.87
21,10,random_forest,4,d-no-skew,0.65,0.85,0.12,0.40,0.73,0.89,0.92
9,10,random_forest,4,daily,0.65,0.85,0.12,0.40,0.72,0.90,0.90


In [19]:
def calibration_table(g, n_bins=8):
    y = g[y_col].astype(int).to_numpy()
    p = np.clip(g[p_col].to_numpy(dtype=float), 1e-15, 1-1e-15)

    # quantile bins (stable counts); duplicates="drop" prevents errors if probs repeat
    bins = pd.qcut(p, q=n_bins, duplicates="drop")

    out = (
        pd.DataFrame({"bin": bins, "y": y, "p": p})
        .groupby("bin", observed=True)
        .agg(
            n=("y", "size"),
            p_mean=("p", "mean"),   # predicted probability avg
            y_rate=("y", "mean"),   # observed frequency
            p_min=("p", "min"),
            p_max=("p", "max"),
        )
        .reset_index(drop=True)
        .sort_values("p_mean")
    )
    return out

cols = ['horizon']
# example: get calibration table for ONE group
key = (2)  # change
g = perf_df.set_index(cols).loc[key].reset_index()
calib_tbl = calibration_table(g, n_bins=10)
round(calib_tbl,2)

,n,p_mean,y_rate,p_min,p_max
0,317,0.09,0.43,0.00,0.20
1,334,0.35,0.39,0.25,0.40
2,184,0.45,0.46,0.45,0.45
3,447,0.52,0.53,0.50,0.55
4,235,0.60,0.69,0.60,0.60
5,228,0.65,0.66,0.65,0.65
6,194,0.70,0.67,0.70,0.70
7,272,0.79,0.71,0.75,0.85
8,525,0.97,0.71,0.90,1.00


In [ ]:
def calibration_table(g, n_bins=8):
    y = g[y_col].astype(int).to_numpy()
    p = np.clip(g[p_col].to_numpy(dtype=float), 1e-15, 1-1e-15)

    # quantile bins (stable counts); duplicates="drop" prevents errors if probs repeat
    bins = pd.qcut(p, q=n_bins, duplicates="drop")

    out = (
        pd.DataFrame({"bin": bins, "y": y, "p": p})
        .groupby("bin", observed=True)
        .agg(
            n=("y", "size"),
            p_mean=("p", "mean"),   # predicted probability avg
            y_rate=("y", "mean"),   # observed frequency
            p_min=("p", "min"),
            p_max=("p", "max"),
        )
        .reset_index(drop=True)
        .sort_values("p_mean")
    )
    return out

# example: get calibration table for ONE group
#key = (10, "xgboost", 6, "daily")  # change
g = perf_df.set_index(gcols).loc[key].reset_index()
calib_tbl = calibration_table(g, n_bins=20)
round(calib_tbl,2)